In [7]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In [3]:
class BrokenNet(nn.Module):
    def __init__(self, depth=8, width=32, in_features=20):
        super().__init__()
        layers = []
        n_in = in_features
        for _ in range(depth):
            layers.append(nn.Linear(n_in, width))
            n_in = width
        self.hidden = nn.ModuleList(layers)
        self.out = nn.Linear(width, 1)

        for layer in self.hidden:
            nn.init.normal_(layer.weight, mean=0.0, std=0.3)
            nn.init.constant_(layer.bias, -2.0)

    def forward(self, x):
        for layer in self.hidden:
            x = torch.relu(layer(x))
        return self.out(x)


def make_toy_classification(n=256, in_features=20, seed=0):
    """A small synthetic binary classification task: a fixed random
    linear boundary in a 20-dimensional input space. Simple enough that
    a correctly trained network of this size should solve it easily.
    """
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, in_features, generator=g)
    true_w = torch.randn(in_features, 1, generator=g)
    logits = x @ true_w
    y = (logits > 0).float().squeeze(-1)
    return x, y

In [5]:
torch.manual_seed(0)
model = BrokenNet()
x, y = make_toy_classification(seed=0)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

print("Training BrokenNet for 50 epochs with plain SGD...")
for epoch in range(50):
    optimizer.zero_grad()
    preds = model(x).squeeze(-1)
    loss = nn.functional.binary_cross_entropy_with_logits(preds, y)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0 or epoch == 49:
        print(f"epoch {epoch:3d}   loss = {loss.item():.4f}")

print("\nFor reference, a model that has learned nothing at all scores")
print("about 0.693 on this loss (ln 2) — check whether the number above")
print("actually moved away from that.")

Training BrokenNet for 50 epochs with plain SGD...
epoch   0   loss = 0.7037
epoch  10   loss = 0.6974
epoch  20   loss = 0.6937
epoch  30   loss = 0.6914
epoch  40   loss = 0.6901
epoch  49   loss = 0.6893

For reference, a model that has learned nothing at all scores
about 0.693 on this loss (ln 2) — check whether the number above
actually moved away from that.


# Diagnosis 

In [11]:
def per_layer_grad_norms(model):
    """Mean absolute gradient of each hidden layer's weight matrix,
    in forward order (layer 0 is closest to the input).
    """
    return [layer.weight.grad.abs().mean().item() for layer in model.hidden]

In [12]:
torch.manual_seed(0)
model = BrokenNet()
x, y = make_toy_classification(seed=0)
preds = model(x).squeeze(-1)

loss = nn.functional.binary_cross_entropy_with_logits(
    preds, y
)
loss.backward()

grad_norms = per_layer_grad_norms(model)

print("Gradient Diagnosis")

for i, grad in enumerate(grad_norms):
    print(f"Layer {i}: mean |gradient| = {grad:.2e}")

Gradient Diagnosis
Layer 0: mean |gradient| = 0.00e+00
Layer 1: mean |gradient| = 0.00e+00
Layer 2: mean |gradient| = 0.00e+00
Layer 3: mean |gradient| = 0.00e+00
Layer 4: mean |gradient| = 0.00e+00
Layer 5: mean |gradient| = 0.00e+00
Layer 6: mean |gradient| = 0.00e+00
Layer 7: mean |gradient| = 0.00e+00
